<a href="https://colab.research.google.com/github/nirlondev/Algoritms/blob/main/googleColab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd, re
from google.colab import files
from io import BytesIO
from datetime import datetime

In [19]:
uploaded = files.upload()
dfs = {}
for f in uploaded.keys():
    dfs[f] = pd.read_excel(BytesIO(uploaded[f]))
    print(f'{f}: {len(dfs[f])} строк')

dfNames, dfPhones = None, None
for name, df in dfs.items():
    if 'ФИО' in df.columns and df['ФИО'].notna().any():
        dfNames = df[['ID', 'ФИО', 'Пол', 'Дата рождения']].copy()
    if 'Номер телефона' in df.columns or 'Телефон' in str(df.columns):
        phoneCol = [c for c in df.columns if 'телефон' in c.lower() or 'phone' in c.lower()][0]
        dfPhones = df[['ID', phoneCol]].copy()
        dfPhones.columns = ['ID', 'Номер телефона']

df = pd.merge(dfNames, dfPhones, on='ID', how='outer')


Saving Данные (ФИО).xlsx to Данные (ФИО) (3).xlsx
Saving Данные (ID, phone).xlsx to Данные (ID, phone) (3).xlsx
Saving Текстовые сообщения.xlsx to Текстовые сообщения (3).xlsx
Данные (ФИО) (3).xlsx: 162 строк
Данные (ID, phone) (3).xlsx: 162 строк
Текстовые сообщения (3).xlsx: 202 строк


In [15]:
vowels, cons = "аеёиоуыэюяАЕЁИОУЫЭЮЯ", "бвгджзйклмнпрстфхцчшщБВГДЖЗЙКЛМНПРСТФХЦЧШЩ"
def countTxt(t):
    if pd.isna(t): return pd.Series([0, 0, 0])
    t = str(t)
    totalChars = len(t)
    punctMarks = sum(c in ".,:;!?-()" for c in t)
    letters = sum(c in vowels or c in cons for c in t)
    return pd.Series([totalChars, punctMarks, letters])

stats = df['ФИО'].apply(countTxt)
print(f'5. Символов в ФИО: мин={stats[0].min()}, макс={stats[0].max()}, ср={stats[0].mean():.1f}')
print(f'6. Знаков препинания: {stats[1].sum()}, Букв (гласные+согласные): {stats[2].sum()}')

5. Символов в ФИО: мин=0, макс=58, ср=10.5
6. Знаков препинания: 0, Букв (гласные+согласные): 1836


In [16]:
def cleanPhone(p):
    if pd.isna(p): return None
    d = re.sub(r'\D', '', str(p))
    if len(d)==11 and d[0]=='8': d = '7'+d[1:]
    if len(d)==10: d = '7'+d
    return '+'+d if len(d)==11 and d[0]=='7' else None

df['phoneBefore'] = df['Номер телефона']
df['phoneInternational'] = df['Номер телефона'].apply(cleanPhone)

In [17]:
def guessGender(name):
    if pd.isna(name): return None
    return 'Ж' if str(name).split()[0].endswith(('а','я')) else 'М'

df['genderCheck'] = df['ФИО'].apply(guessGender)
df['Пол'] = df.apply(lambda r: r['genderCheck'] if pd.isna(r.get('Пол')) else r['Пол'], axis=1)

def calcAge(d):
    if pd.isna(d): return None
    try:
        b = pd.to_datetime(d)
        return datetime.now().year - b.year - ((datetime.now().month,datetime.now().day) < (b.month,b.day))
    except: return None

df['Возраст'] = df['Дата рождения'].apply(calcAge)

In [18]:
out = df[['ID','ФИО','phoneBefore','phoneInternational','Пол','Возраст']].copy()
out.columns = ['ID','ФИО','Номер_до_форматирования','Номер_международный','Пол','Возраст']
out = out.dropna(subset=['ФИО','Номер_международный','Пол','Возраст'])

out.to_csv('finalDirectory.csv', index=False, encoding='utf-8-sig')
out.to_excel('finalDirectory.xlsx', index=False)
files.download("finalDirectory.xlsx")
files.download('finalDirectory.csv')
print(f'Строк в CSV: {len(out)}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Строк в CSV: 69
